# CH10 — Diccionarios (parte 2): hacia las tablas hash

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 10.

Continuación de `ch10_teoria.ipynb`, donde construimos `UnsortedTableMap`: un diccionario que guarda sus entradas `[clave, valor]` en una lista y busca recorriéndola, en $O(n)$.

**Objetivo:** ver cómo cambia un diccionario cuando, en lugar de **buscar** la casilla de una clave, la **calculamos**.

**Contenido**

1. Cambiar cómo se busca: la función de índice
2. Un ejemplo pequeño: $N = 10$
3. La versión modificada: `IndexedTableMap`
4. El millón de entradas
5. ¿Por qué esto todavía no es una tabla hash?
6. Resumen: tablas hash, separate chaining y linear probing
7. Experimento: colisiones con palabras reales

## 1. Cambiar **cómo** se busca

<p align="center">
  <img src='https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRmHMY0D2_u1EjiwUgNSzx-G_0GeuRMEP6YCbk31MSlww&s=10'>
</p>

En `UnsortedTableMap` el método `_buscar(k)` recorre la lista comparando clave por clave. Con pocas entradas no se nota, pero imagina un diccionario con **un millón** de entradas:

- claves: los enteros `1, 2, ..., 1_000_000`
- buscar una clave que está al final cuesta **un millón de comparaciones**, y en promedio medio millón.

La pregunta de esta sección es:

> **¿Y si cada clave tuviera su propia casilla, y pudiéramos *calcular* en qué casilla está sin recorrer nada?**

### La función de índice

Reservamos una lista de tamaño $N$ (una casilla por clave posible) y definimos una **función de índice** $h$ que a cada clave le asigna una casilla:

$$h : \{1, 2, \dots, N\} \longrightarrow \{0, 1, \dots, N-1\}$$

La entrada `[k, v]` se guarda en `self._table[h(k)]`. Para buscar `k` ya no se recorre nada: se calcula `h(k)` y se mira **una sola casilla**.

Para que funcione, $h$ debe ser una **permutación**: claves distintas van a casillas distintas (ninguna casilla es compartida) y todas las casillas se usan.

La permutación más simple es $h(k) = k - 1$. Aquí usamos a propósito una permutación **revuelta al azar**, para dejar claro que el orden de las casillas no importa: lo único que importa es que **cada clave tenga su propia casilla y que podamos calcularla**. Para que todos veamos la misma permutación, la generamos con una **semilla fija**.

## 2. Un ejemplo pequeño: $N = 10$

Antes del millón, veámoslo con 10 claves.

In [1]:
import random

def crear_funcion_indice(N, semilla=2028):
    # Retorna una función h que asigna a cada clave 1..N una casilla 0..N-1 distinta.
    perm = list(range(N))                 # [0, 1, ..., N-1]
    random.Random(semilla).shuffle(perm)  # la revuelve; misma semilla -> misma permutación

    def h(k):
        return perm[k - 1]
    return h

h10 = crear_funcion_indice(10)
for k in range(1, 11):
    print(f'clave {k:2} -> casilla {h10(k)}')
# clave  1 -> casilla 3
# clave  2 -> casilla 7
# clave  3 -> casilla 4
# clave  4 -> casilla 1
# clave  5 -> casilla 9
# clave  6 -> casilla 6
# clave  7 -> casilla 8
# clave  8 -> casilla 0
# clave  9 -> casilla 2
# clave 10 -> casilla 5

clave  1 -> casilla 3
clave  2 -> casilla 7
clave  3 -> casilla 4
clave  4 -> casilla 1
clave  5 -> casilla 9
clave  6 -> casilla 6
clave  7 -> casilla 8
clave  8 -> casilla 0
clave  9 -> casilla 2
clave 10 -> casilla 5


In [2]:
# Así queda la tabla si guardamos las claves 1..10 con valor k*k
tabla = [None] * 10
for k in range(1, 11):
    tabla[h10(k)] = [k, k * k]

for j, entrada in enumerate(tabla):
    print(f'casilla {j}: {entrada}')
# casilla 0: [8, 64]
# casilla 1: [4, 16]
# casilla 2: [9, 81]
# casilla 3: [1, 1]
# casilla 4: [3, 9]
# casilla 5: [10, 100]
# casilla 6: [6, 36]
# casilla 7: [2, 4]
# casilla 8: [7, 49]
# casilla 9: [5, 25]

# ¿Dónde está la clave 7? No hay que recorrer: se calcula h10(7)
print(h10(7), tabla[h10(7)])     # 8 [7, 49]

casilla 0: [8, 64]
casilla 1: [4, 16]
casilla 2: [9, 81]
casilla 3: [1, 1]
casilla 4: [3, 9]
casilla 5: [10, 100]
casilla 6: [6, 36]
casilla 7: [2, 4]
casilla 8: [7, 49]
casilla 9: [5, 25]
8 [7, 49]


## 3. La versión modificada: `IndexedTableMap`

Tomamos `UnsortedTableMap` y cambiamos `_buscar`. Pero al cambiar **dónde** vive cada entrada, hay otros métodos que también deben cambiar:

| Método | `UnsortedTableMap` | `IndexedTableMap` |
|---|---|---|
| `__init__` | `self._table = []` | `self._table = [None] * N` (todas las casillas existen desde el inicio, vacías) |
| `_buscar(k)` | recorre la lista: $O(n)$ | mira **solo** la casilla `h(k)`: $O(1)$ |
| `__setitem__` | si no está, `append` al final | guarda en la casilla `h(k)` |
| `__delitem__` | `pop(j)` (corre las demás entradas) | `self._table[j] = None` (las demás **no se pueden mover**: su casilla es fija) |
| `__len__` | `len(self._table)` | un contador `self._n` (la lista siempre mide $N$, aunque esté casi vacía) |
| `__iter__` | recorre la lista | recorre la lista **saltando** los `None` |

**Observa:**

- `_buscar` **no compara** `k` con otras claves. Como $h$ es una permutación, la única clave que puede estar en la casilla `h(k)` es `k` misma: o está ahí, o no está en el diccionario.
- Se pierde el **orden de inserción**: `for k in M` recorre las claves en el orden de las casillas.

In [3]:
class IndexedTableMap:
    # Diccionario para claves enteras 1..N: la entrada [k, v] vive en la casilla h(k).

    def __init__(self, N, h):
        self._table = [None] * N          # una casilla por cada clave posible
        self._h = h                       # función de índice: clave -> casilla
        self._n = 0                       # número de entradas guardadas

    # ---------- auxiliar: ESTE es el cambio importante ----------
    def _buscar(self, k):
        # Retorna la casilla de k si k está guardada, o -1 si no.
        if not (1 <= k <= len(self._table)):   # clave fuera de rango: no puede estar
            return -1
        j = self._h(k)                         # se CALCULA la casilla, no se recorre
        if self._table[j] is None:             # casilla vacía: k no está
            return -1
        return j

    # ---------- núcleo ----------
    def __len__(self):
        return self._n

    def __getitem__(self, k):
        j = self._buscar(k)
        if j == -1:
            raise KeyError(k)
        return self._table[j][1]

    def __setitem__(self, k, v):
        if not (1 <= k <= len(self._table)):
            raise KeyError(f'{k} no está entre 1 y {len(self._table)}')
        j = self._h(k)
        if self._table[j] is None:        # clave nueva
            self._table[j] = [k, v]
            self._n += 1
        else:                             # ya existe: reemplaza
            self._table[j][1] = v

    def __delitem__(self, k):
        j = self._buscar(k)
        if j == -1:
            raise KeyError(k)
        self._table[j] = None             # vaciar la casilla, sin mover a nadie
        self._n -= 1

    def __contains__(self, k):
        return self._buscar(k) != -1

    def __iter__(self):
        for entrada in self._table:
            if entrada is not None:
                yield entrada[0]

    def __repr__(self):
        return '{' + ', '.join(f'{e[0]!r}: {e[1]!r}' for e in self._table if e is not None) + '}'

In [4]:
# --- Prueba con N = 10 ---
M = IndexedTableMap(10, h10)
M[7] = 'siete'
M[2] = 'dos'
M[10] = 'diez'
print(len(M), M[2])    # 3 dos
print(M._table)        # [None, None, None, None, None, [10, 'diez'], None, [2, 'dos'], [7, 'siete'], None]

M[2] = 'DOS'           # reemplaza, no agrega
print(len(M), M[2])    # 3 DOS

del M[7]
print(7 in M, 10 in M, 99 in M)   # False True False
print(list(M))         # [10, 2]   <- orden de las casillas, NO de inserción

3 dos
[None, None, None, None, None, [10, 'diez'], None, [2, 'dos'], [7, 'siete'], None]
3 DOS
False True False
[10, 2]


## 4. El millón de entradas

Ahora sí: $N = 1\,000\,000$. Construimos los dos diccionarios con las mismas entradas `k -> k*k` y comparamos cuánto tarda buscar.

Para la versión no ordenada usamos una lista de entradas y una búsqueda lineal (lo mismo que hace `_buscar` en `UnsortedTableMap`), así esta celda funciona aunque tu clase no esté terminada.

*(Construir cada estructura toma un par de segundos.)*

In [5]:
N = 1000000
h = crear_funcion_indice(N)            # misma semilla: todos tenemos la misma permutación
print(h(1), h(2), h(3), h(N))          # 685003 904654 162457 372357

# ¿De verdad es una permutación? Claves distintas -> casillas distintas
print(len({h(k) for k in range(1, N + 1)}) == N)   # True

685003 904654 162457 372357
True


In [6]:
# --- Las dos estructuras con el mismo millón de entradas ---
lista_no_ordenada = [[k, k * k] for k in range(1, N + 1)]   # como UnsortedTableMap._table

def buscar_lineal(tabla, k):
    # El _buscar de UnsortedTableMap: recorre comparando clave por clave.
    for j in range(len(tabla)):
        if tabla[j][0] == k:
            return j
    return -1


In [7]:
D = IndexedTableMap(N, h)
for k in range(1, N + 1):
    D[k] = k * k

print(len(lista_no_ordenada), len(D))   # 1000000 1000000

1000000 1000000


In [8]:
import time

claves = [1, 500_000, 999_999, N]      # una al inicio, una a la mitad, dos al final

for k in claves:
    t0 = time.perf_counter()
    j = buscar_lineal(lista_no_ordenada, k)
    t_lineal = time.perf_counter() - t0

    t0 = time.perf_counter()
    j2 = D._buscar(k)
    t_indice = time.perf_counter() - t0

    print(f'clave {k:>9,}: lineal revisa {j + 1:>9,} casillas en {t_lineal*1000:8.3f} ms | '
          f'índice revisa 1 casilla (la {j2:>7,}) en {t_indice*1000:.4f} ms')

clave         1: lineal revisa         1 casillas en    0.003 ms | índice revisa 1 casilla (la 685,003) en 0.0031 ms
clave   500,000: lineal revisa   500,000 casillas en   27.668 ms | índice revisa 1 casilla (la 848,838) en 0.0097 ms
clave   999,999: lineal revisa   999,999 casillas en   44.604 ms | índice revisa 1 casilla (la 868,516) en 0.0087 ms
clave 1,000,000: lineal revisa 1,000,000 casillas en   71.684 ms | índice revisa 1 casilla (la 372,357) en 0.0082 ms


**Lo que se observa:**

- La búsqueda lineal tarda **más mientras más atrás está la clave**: para la clave `1_000_000` revisa el millón de casillas.
- La búsqueda con la función de índice tarda **lo mismo para cualquier clave**: un cálculo y una casilla. Es $O(1)$ y **no depende de $N$**.

## 5. ¿Por qué esto todavía no es una tabla hash?

`IndexedTableMap` funciona porque hicimos dos trampas:

1. **Las claves son exactamente `1..N`.** ¿Qué pasa si las claves son cédulas (números de 10 dígitos), nombres o palabras? No podemos reservar una casilla para **cada cédula posible**: serían $10^{10}$ casillas para guardar quizá unos cientos de estudiantes.
2. **$h$ es una tabla guardada**: para calcular `h(k)` consultamos la lista `perm`, que ocupa tanta memoria como el diccionario mismo.

Una **tabla hash** conserva la idea (calcular la casilla en vez de buscarla) pero:

- usa una tabla de tamaño $N$ **pequeño**, proporcional al número de entradas, no al número de claves posibles;
- usa una **función hash** que se calcula con aritmética a partir de la clave, por ejemplo `hash(k) % N`, sin guardar nada.

El precio es que $h$ **ya no puede ser una permutación**: hay muchas más claves posibles que casillas, así que dos claves distintas pueden caer en la misma casilla. Eso es una **colisión**, y manejar las colisiones es el tema central de las tablas hash.

In [9]:
# Colisiones: tabla de 10 casillas, claves que son cédulas
N_chico = 10
for cedula in [1032456789, 52347719, 1019283745, 80123459]:
    print(cedula, '-> casilla', hash(cedula) % N_chico)
# 1032456789 -> casilla 9
# 52347719 -> casilla 9      <- ¡misma casilla que la anterior: colisión!
# 1019283745 -> casilla 5
# 80123459 -> casilla 9      <- y otra más

1032456789 -> casilla 9
52347719 -> casilla 9
1019283745 -> casilla 5
80123459 -> casilla 9


## 6. Resumen: tablas hash y manejo de colisiones

### ¿Qué es una tabla hash?

Una **tabla hash** es un diccionario que guarda sus entradas en una lista de $N$ casillas (en Goodrich, un *bucket array*) y usa una **función hash** para calcular en qué casilla va cada clave:

$$h(k) = \texttt{hash}(k) \bmod N$$

- `hash(k)` convierte cualquier clave inmutable (`int`, `str`, `tuple`...) en un entero (*hash code*).
- `% N` lo lleva al rango de casillas `0..N-1` (*compresión*).
- El **factor de carga** $\lambda = n / N$ mide qué tan llena está la tabla ($n$ = número de entradas). Cuando $\lambda$ supera un límite, se crea una tabla más grande (por ejemplo de tamaño $2N$) y se **reinsertan** todas las entradas, igual que una lista dinámica crece al hacer `append`.

Como hay muchas más claves posibles que casillas, las **colisiones** son inevitables. Hay dos formas clásicas de resolverlas. Las veremos con las cédulas de la celda anterior, en una tabla de $N = 10$, insertadas en este orden:

| Cédula | `hash(k) % 10` |
|---|---|
| 1032456789 | 9 |
| 52347719 | 9 |
| 1019283745 | 5 |
| 80123459 | 9 |

### Separate chaining (encadenamiento)

Cada casilla guarda una **lista** (un *bucket*) con **todas** las entradas cuya clave cae en esa casilla. Una colisión simplemente agrega la entrada al bucket.

```
casilla 0: []
  ...
casilla 5: [ [1019283745, v] ]
  ...
casilla 9: [ [1032456789, v], [52347719, v], [80123459, v] ]
```

- **Buscar `k`:** calcular $j = h(k)$ y recorrer **solo** el bucket de la casilla $j$. Es una búsqueda lineal, pero en una lista muy corta (en promedio $\lambda$ entradas). De hecho, cada bucket puede ser un `UnsortedTableMap`.
- **Insertar / borrar:** igual, dentro del bucket de la casilla $j$.
- **Costo esperado:** $O(1 + \lambda)$. Se suele mantener $\lambda < 0.9$.

### Linear probing (sondeo lineal)

Cada casilla guarda **a lo sumo una** entrada. Si la casilla $j = h(k)$ está ocupada por otra clave, se prueba la siguiente, $j+1$, luego $j+2$, ..., **dando la vuelta** al final de la tabla: $(j + i) \bmod N$.

```
1032456789 -> 9 libre                    -> casilla 9
52347719   -> 9 ocupada, 0 libre         -> casilla 0
1019283745 -> 5 libre                    -> casilla 5
80123459   -> 9 ocupada, 0 ocupada, 1    -> casilla 1

casilla:  0          1          2 3 4  5           6 7 8  9
       [52347719, 80123459, -, -, -, 1019283745, -, -, -, 1032456789]
```

- **Buscar `k`:** empezar en $h(k)$ y avanzar hasta encontrar `k` o una casilla **vacía**. La casilla vacía significa que `k` no está.
- **Borrar:** no se puede dejar la casilla en `None`, porque cortaría la búsqueda de las claves que pasaron por ahí. Por ejemplo, si se borra `52347719` y la casilla 0 queda vacía, la búsqueda de `80123459` se detendría en 0 y diría que no está. Se usa una **marca especial** ("disponible", `_AVAIL` en Goodrich): la búsqueda la salta y la inserción puede reutilizarla.
- **Problema:** las entradas tienden a formar **bloques contiguos** (*clustering*), y los bloques hacen más largos los sondeos. Por eso se mantiene $\lambda < 0.5$.

### Comparación

| | Separate chaining | Linear probing |
|---|---|---|
| Qué guarda cada casilla | una lista (bucket) de entradas | a lo sumo una entrada |
| Colisión | se agrega al bucket | se prueba la casilla siguiente |
| Borrar | se quita del bucket | se deja la marca "disponible" |
| Factor de carga típico | $\lambda < 0.9$ | $\lambda < 0.5$ |
| Memoria extra | las listas de cada bucket | ninguna, pero necesita más casillas vacías |
| Costo esperado de buscar / insertar / borrar | $O(1)$ | $O(1)$ |
| Peor caso (todas las claves en la misma casilla) | $O(n)$ | $O(n)$ |

El `dict` de Python es una tabla hash con *open addressing*, la misma familia que linear probing: una entrada por casilla. La diferencia es que no prueba la casilla siguiente, sino una secuencia de casillas que depende de los bits del hash.

## 7. Experimento: colisiones con palabras reales

Para terminar, contamos colisiones con datos reales: unas **80 000 palabras del español**, tomadas de la lista pública [javierarce/palabras](https://github.com/javierarce/palabras) en GitHub.

El plan:

1. Descargar la lista de palabras.
2. Implementar una tabla hash con **capacidad fija** $N$ (nunca crece) y separate chaining, que **cuente las colisiones**.
3. Insertar todas las palabras con dos funciones hash distintas y comparar.

**¿Qué contamos como colisión?** Cada vez que una clave **nueva** cae en una casilla que **ya tenía** alguna clave. Así, si $n$ claves ocupan $c$ casillas distintas, hubo exactamente $n - c$ colisiones.

In [10]:
import urllib.request

URL = 'https://raw.githubusercontent.com/javierarce/palabras/master/listado-general.txt'
texto = urllib.request.urlopen(URL).read().decode('utf-8')
palabras = texto.split()

print(len(palabras))                 # 80383
print(palabras[:5])                  # ['a', 'aaronita', 'aarónico', 'aba', 'ababa']
print(palabras[-3:])                 # ['zócalo', 'zóster', 'zúa']
print(len(set(palabras)) == len(palabras))   # True: no hay palabras repetidas

80383
['a', 'aaronita', 'aarónico', 'aba', 'ababa']
['zócalo', 'zóster', 'zúa']
True


### 7.1 Dos funciones hash para cadenas

Una función hash para cadenas convierte la palabra en un entero (*hash code*) y luego lo lleva al rango `0..N-1` con `% N`.

- **Suma de códigos:** suma los códigos Unicode de las letras (`ord(c)`). Es la idea más simple, pero es mala: **ignora el orden** de las letras, así que `'roma'`, `'amor'`, `'mora'` y `'ramo'` dan el mismo valor. Además las sumas son números pequeños (pocas casillas posibles).
- **Polinomial** (Goodrich, sección 10.2.1): trata la palabra como un polinomio evaluado en una constante $a$:
$$x_0\,a^{m-1} + x_1\,a^{m-2} + \dots + x_{m-2}\,a + x_{m-1}$$
  donde $x_i$ es el código de la letra $i$. Se calcula con la **regla de Horner**: `h = h * a + ord(c)`. Ahora el orden importa. Goodrich reporta que $a = 33, 37, 39, 41$ dan muy pocas colisiones en palabras del inglés. Para que el número no crezca sin límite, nos quedamos con los últimos 32 bits (`& 0xFFFFFFFF`), como haría un entero de 32 bits.

**¿Por qué no usar `hash()` de Python?** Porque para cadenas, Python **cambia** `hash()` en cada sesión (por seguridad), así que cada uno vería un número distinto de colisiones. Con nuestras funciones, todos vemos lo mismo.

In [11]:
def hash_suma(s):
    # Suma de los códigos de las letras: ignora el orden.
    total = 0
    for c in s:
        total += ord(c)
    return total

def hash_polinomial(s, a=33):
    # Polinomio en a evaluado con la regla de Horner, recortado a 32 bits.
    h = 0
    for c in s:
        h = (h * a + ord(c)) & 0xFFFFFFFF
    return h

for p in ['roma', 'amor', 'mora', 'ramo']:
    print(f'{p:5} suma = {hash_suma(p):4}   polinomial = {hash_polinomial(p):>9}')
# roma  suma =  431   polinomial =   4221391
# amor  suma =  431   polinomial =   3608367
# mora  suma =  431   polinomial =   4041871
# ramo  suma =  431   polinomial =   4206159

roma  suma =  431   polinomial =   4221391
amor  suma =  431   polinomial =   3608367
mora  suma =  431   polinomial =   4041871
ramo  suma =  431   polinomial =   4206159


### 7.2 Para la clase: implementa `TablaHashFija`

`TablaHashFija` usa separate chaining como `ChainHashMap` de Goodrich, con dos diferencias:

- **No tiene `_resize`**: la capacidad $N$ se fija al crearla y nunca cambia, aunque $\lambda = n/N$ crezca.
- **Cuenta colisiones** en `self.colisiones`, y trae métodos para medir la tabla: casillas ocupadas y lista más larga.

Cada casilla es una lista de entradas `[clave, valor]`, la misma idea de `UnsortedTableMap`.

`__init__`, `_casilla` y `resumen` ya están dados. Completa cada método marcado con `pass`:

- `__len__()`: número de entradas guardadas.
- `__setitem__(k, v)`: busca `k` **solo** en la lista de la casilla `self._casilla(k)`. Si ya está, reemplaza su valor. Si no está, agrégala al final de esa lista y aumenta `self._n`. Si la lista **ya tenía** alguna entrada antes de agregar, aumenta `self.colisiones`.
- `__getitem__(k)`: retorna el valor de `k` buscando en la lista de su casilla. Si no está, `raise KeyError(k)`.
- `__contains__(k)`: `True` si `k` está en la lista de su casilla.
- `capacidad()`: el número de casillas $N$.
- `casillas_ocupadas()`: cuántas casillas tienen la lista no vacía.
- `lista_mas_larga()`: la longitud de la lista más larga.

**Ojo:** ningún método debe recorrer **toda** la tabla para buscar una clave; eso es lo que la tabla hash evita. Solo `casillas_ocupadas` y `lista_mas_larga` recorren todas las casillas, porque miden la tabla completa.

In [12]:
class TablaHashFija:
    # Tabla hash con separate chaining y capacidad N fija. Cuenta las colisiones.

    def __init__(self, N, funcion_hash):
        self._table = [[] for _ in range(N)]   # N casillas, cada una con su lista
        self._f = funcion_hash                 # clave -> entero (hash code)
        self._n = 0
        self.colisiones = 0

    def _casilla(self, k):
        return self._f(k) % len(self._table)   # compresión: llevar a 0..N-1

    # ---------- núcleo ----------
    def __len__(self):
        # len(T): número de entradas
        pass

    def __setitem__(self, k, v):
        # T[k] = v  (inserta o reemplaza; si k es nueva y su casilla ya tenía algo, suma una colisión)
        pass

    def __getitem__(self, k):
        # T[k]  (KeyError si no existe)
        pass

    def __contains__(self, k):
        # k in T
        pass

    # ---------- medidas ----------
    def capacidad(self):
        # número de casillas N
        pass

    def casillas_ocupadas(self):
        # número de casillas cuya lista no está vacía
        pass

    def lista_mas_larga(self):
        # longitud de la lista más larga
        pass

    # ---------- dado ----------
    def resumen(self):
        n, N = len(self), self.capacidad()
        return (f'n = {n}, N = {N}, λ = {n / N:.2f} | colisiones = {self.colisiones} | '
                f'casillas ocupadas = {self.casillas_ocupadas()} | lista más larga = {self.lista_mas_larga()}')

In [13]:
# --- Prueba pequeña: los anagramas chocan con hash_suma ---
T = TablaHashFija(11, hash_suma)
for p in ['roma', 'amor', 'mora', 'ramo', 'sol']:
    T[p] = len(p)
print(T.colisiones)          # 3: amor, mora y ramo caen donde ya estaba roma
print(T['mora'], 'ramo' in T, 'luna' in T)   # 4 True False
print(T._table[431 % 11])    # [['roma', 4], ['amor', 4], ['mora', 4], ['ramo', 4]]

0
None False False
[]


### 7.3 Las 80 000 palabras

Usamos $N = 100\,003$ casillas (un número primo, un poco más que el número de palabras), así que $\lambda \approx 0.8$.

**¿Cuántas colisiones son "normales"?** Aun con una función hash perfecta, que reparte las claves **al azar** entre las casillas, hay colisiones. Si se insertan $n$ claves al azar en $N$ casillas, el número esperado de casillas ocupadas es

$$c = N\left(1 - \left(1 - \tfrac{1}{N}\right)^{n}\right)$$

así que se esperan $n - c$ colisiones. Ese es el punto de referencia para juzgar una función hash: una buena función se acerca a ese número; una mala lo supera de lejos.

In [14]:
def colisiones_esperadas(n, N):
    # Colisiones esperadas si las n claves cayeran al azar en las N casillas.
    ocupadas = N * (1 - (1 - 1 / N) ** n)
    return n - ocupadas

N = 100_003
n = len(palabras)
print(f'Referencia (al azar): {colisiones_esperadas(n, N):.0f} colisiones')   # 25143
print()

for nombre, f in [('suma', hash_suma), ('polinomial', hash_polinomial)]:
    T = TablaHashFija(N, f)
    for i, p in enumerate(palabras):
        T[p] = i
    print(f'{nombre:11}', T.resumen())
# suma        n = 80383, N = 100003, λ = 0.80 | colisiones = 78613 | casillas ocupadas = 1770 | lista más larga = 271
# polinomial  n = 80383, N = 100003, λ = 0.80 | colisiones = 25177 | casillas ocupadas = 55206 | lista más larga = 7

Referencia (al azar): 25143 colisiones



TypeError: 'NoneType' object cannot be interpreted as an integer

**Lo que se observa:**

- Con **suma de códigos**, las 80 383 palabras quedan en apenas 1 770 de las 100 003 casillas: las sumas de letras son números pequeños (en esta lista nunca pasan de 2 500) y se repiten muchísimo. Casi todas las inserciones son colisiones, y la lista más larga tiene 271 palabras: buscar en ella es volver a la búsqueda lineal.
- Con la **polinomial**, hay 25 177 colisiones contra 25 143 esperadas al azar, y la lista más larga tiene solo 7 palabras. Eso es lo que hace que las operaciones cuesten $O(1)$ en promedio.

### 7.4 ¿Y si cambiamos la capacidad?

Con $N$ fijo, mientras más pequeña la tabla, más alto el factor de carga y más colisiones. Por eso `HashMapBase` de Goodrich duplica $N$ cuando $\lambda$ pasa de $0.5$.

In [ ]:
print(f'{"N":>8} {"λ":>6} {"colisiones":>11} {"esperadas":>10} {"lista más larga":>16}')
for N in [10_007, 40_009, 80_021, 160_001, 320_009]:
    T = TablaHashFija(N, hash_polinomial)
    for i, p in enumerate(palabras):
        T[p] = i
    print(f'{N:>8} {len(T) / N:>6.2f} {T.colisiones:>11} {colisiones_esperadas(len(T), N):>10.0f} {T.lista_mas_larga():>16}')
#        N      λ  colisiones  esperadas  lista más larga
#    10007   8.03       70376      70379               19
#    40009   2.01       45674      45739               10
#    80021   1.00       29754      29667                7
#   160001   0.50       17091      17196                6
#   320009   0.25        9297       9301                5

Las colisiones siempre quedan muy cerca de las esperadas al azar: la función polinomial reparte bien en todos los casos. Lo que cambia es $\lambda$. Con $\lambda = 8$ cada búsqueda recorre listas de unas 8 palabras en promedio; con $\lambda = 0.5$, casi siempre una sola.

### 7.5 ¿Por qué $N$ suele ser primo?

Mira la regla de Horner: `h = h * 33 + ord(c)`. Si $N = 33^3$, todo lo que se multiplicó por $33^3$ o más desaparece al hacer `% N`. Es decir, **la casilla depende solo de las últimas 3 letras**. Todas las palabras terminadas en *-ción*, *-mente*, *-ado*... que comparten las 3 últimas letras chocan entre sí.

Comparemos $N = 33^3 = 35\,937$ con un primo cercano, $35\,951$:

In [ ]:
for N in [33 ** 3, 35_951]:
    T = TablaHashFija(N, hash_polinomial)
    for i, p in enumerate(palabras):
        T[p] = i
    print(f'N = {N}: colisiones = {T.colisiones} (esperadas {colisiones_esperadas(len(T), N):.0f}), '
          f'lista más larga = {T.lista_mas_larga()}')
# N = 35937: colisiones = 53979 (esperadas 48284), lista más larga = 150
# N = 35951: colisiones = 48306 (esperadas 48275), lista más larga = 9

Con $N = 33^3$ hay más colisiones de las esperadas y, sobre todo, una lista de **150** palabras. Un $N$ primo no comparte factores con $a$, y todas las letras de la palabra cuentan para elegir la casilla.

**Para pensar:**

1. Cambia `a=33` por `a=1` en `hash_polinomial`. ¿Qué función se obtiene? Compara sus colisiones con las de `hash_suma`.
2. Prueba $a = 37, 39, 41$ con $N = 100\,003$. ¿Cambia mucho el número de colisiones?
3. ¿Qué crees que pasa con $N = 33^2$? ¿Y con $N = 33$? Pruébalo y mira cuáles palabras quedan en la lista más larga.